In [4]:
!pip install tensorflow==2.13.0  wandb librosa matplotlib scikit-learn

  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.0/24.0 MB 401.8 kB/s  0:00:57m0:00:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 109.8 kB/s  0:00:16 eta 0:00:02
Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
Using cached typing_inspection-0.4.2-py3-none-any.whl (14 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [wandb]32m5/6 [wandb]ic]k]

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [12]:

!pip install numpy
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing import image_dataset_from_directory
from sklearn.utils import class_weight
import matplotlib.pyplot as plt
import wandb
from wandb.keras import WandbCallback


RecursionError: maximum recursion depth exceeded

In [ ]:
# 1. Initialize WandB
wandb.init(
    project="Saka-RHD-Detection",
    config={
        "architecture": "MobileNetV2",
        "dataset": "PhysioNet-CirCOR-Hybrid",
        "learning_rate": 0.0005,
        "epochs": 30,
        "batch_size": 32,
        "img_size": (128, 128)
    }
)
config = wandb.config


In [ ]:
# 2. Paths
DATA_PATH = "../data/processed/heart_grades_classified"
MODEL_SAVE_PATH = "../exported_models/saka_v1.h5"

In [ ]:
# 3. Load Datasets
train_ds = image_dataset_from_directory(
    DATA_PATH,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=config.img_size,
    batch_size=config.batch_size,
    label_mode='int'
)

val_ds = image_dataset_from_directory(
    DATA_PATH,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=config.img_size,
    batch_size=config.batch_size,
    label_mode='int'
)

class_names = train_ds.class_names
print(f"Classes found: {class_names}")


In [ ]:
# 4. Handle Class Imbalance

# calculate weights so the model pays 20x more attention to Grade 2.
y_train = np.concatenate([y for x, y in train_ds], axis=0)
weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights_dict = dict(enumerate(weights))
print(f"Calculated Class Weights: {class_weights_dict}")

# 5. Data Augmentation (Crucial for the minority classes)
data_augmentation = models.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])


In [ ]:
# 6. Build Model using MobileNetV2 Transfer Learning

base_model = MobileNetV2(input_shape=(128, 128, 3), include_top=False, weights='imagenet')
base_model.trainable = False 

model = models.Sequential([
    layers.Input(shape=(128, 128, 3)),
    data_augmentation,
    layers.Lambda(lambda x: tf.keras.applications.mobilenet_v2.preprocess_input(x)),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5), 
    layers.Dense(4, activation='softmax') 
])

model.compile(
    optimizer=optimizers.Adam(learning_rate=config.learning_rate),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.SparseTopKCategoricalAccuracy(k=2, name='top_2_accuracy')]
)

In [ ]:
# 7. Training
print("Training started")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=config.epochs,
    class_weight=class_weights_dict,
    callbacks=[
        WandbCallback(),
        tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
        tf.keras.callbacks.ModelCheckpoint(MODEL_SAVE_PATH, save_best_only=True)
    ]
)


In [ ]:
# 8. Evaluation & Final Logging
print(" Training Finished. Evaluating...")
loss, acc, top2 = model.evaluate(val_ds)
wandb.log({"final_val_accuracy": acc, "final_val_loss": loss})

In [ ]:





# 9. Save as TFLite for the Mobile App / Edge
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
with open("../exported_models/saka_v1.tflite", "wb") as f:
    f.write(tflite_model)

print(f"Model exported to {MODEL_SAVE_PATH} and TFLite version.")
wandb.finish()